# Proportional Clearing

## From balance sheets to clearing payments

For $n$ institutions, let total nominal liabilities be $\overline p\in\mathbb R_+^n$, external assets be $e\in\mathbb R_+^n$, and relative liabilities be $\Pi\in\mathbb R_+^{n\times n}$. The entry $\Pi_{ij}$ is the share of institution $i$'s total payment sent to institution $j$. The missing row mass is paid outside the network.

The basic proportional clearing equation is

$$
p=\min\!\left(\overline p,\ e+\Pi^{\mathsf T}p\right),
$$

where the minimum is taken component by component. The vector $p$ contains total payments, $\overline p$ contains promised payments, $e$ contains assets from outside the network, and $\Pi^{\mathsf T}p$ contains payments received from other institutions. Limited liability prevents an institution from paying more than its available resources, while the upper bound prevents it from paying more than it owes.

### Small network examples

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.optimize import linprog


BOOK_ROOT = Path.cwd().parents[1]

import RiskLabAI
from RiskLabAI.optimization import eisenberg_noe_clearing

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "28_network_clearing"
TABLES = BOOK_ROOT / "outputs" / "tables" / "28_network_clearing"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "28_network_clearing"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def ring_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    for debtor in range(size):
        matrix[debtor, (debtor + 1) % size] = internal_share
    return matrix


def complete_matrix(size, internal_share=0.8):
    matrix = np.full((size, size), internal_share / (size - 1))
    np.fill_diagonal(matrix, 0)
    return matrix


def core_periphery_matrix(size, internal_share=0.8, core_size=4):
    matrix = np.zeros((size, size))
    core = np.arange(core_size)
    periphery = np.arange(core_size, size)
    for debtor in range(size):
        if debtor < core_size:
            other_core = core[core != debtor]
            matrix[debtor, other_core] = 0.4 / len(other_core)
            matrix[debtor, periphery] = 0.4 / len(periphery)
        else:
            matrix[debtor, core] = 0.64 / len(core)
            other_periphery = periphery[periphery != debtor]
            matrix[debtor, other_periphery] = 0.16 / len(other_periphery)
    return matrix * (internal_share / 0.8)


def community_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    half = size // 2
    for debtor in range(size):
        own = np.arange(0, half) if debtor < half else np.arange(half, size)
        other = np.arange(half, size) if debtor < half else np.arange(0, half)
        own = own[own != debtor]
        matrix[debtor, own] = 0.68 / len(own)
        matrix[debtor, other] = 0.12 / len(other)
    return matrix * (internal_share / 0.8)


def greatest_iteration(external_assets, nominal_liabilities, relative_liabilities, tolerance=1e-12, maximum_steps=10000):
    payments = np.asarray(nominal_liabilities, dtype=float).copy()
    history = [payments.copy()]
    for _ in range(maximum_steps):
        updated = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
        history.append(updated.copy())
        if np.max(np.abs(updated - payments)) <= tolerance:
            return updated, np.asarray(history)
        payments = updated
    raise RuntimeError("Monotone iteration did not converge.")


def direct_linear_program(external_assets, nominal_liabilities, relative_liabilities):
    result = linprog(
        -np.ones(len(nominal_liabilities)),
        A_ub=np.eye(len(nominal_liabilities)) - relative_liabilities.T,
        b_ub=np.minimum(external_assets, nominal_liabilities),
        bounds=np.column_stack((np.zeros(len(nominal_liabilities)), nominal_liabilities)),
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return np.asarray(result.x)


def clearing_metrics(external_assets, nominal_liabilities, relative_liabilities):
    payments = eisenberg_noe_clearing(external_assets, nominal_liabilities, relative_liabilities)
    fixed = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
    unpaid = np.maximum(nominal_liabilities - payments, 0)
    return {
        "payments": payments,
        "unpaid": unpaid,
        "total payments": float(payments.sum()),
        "total unpaid": float(unpaid.sum()),
        "default count": int(np.sum(unpaid > 1e-8)),
        "payment ratio": float(payments.sum() / nominal_liabilities.sum()) if nominal_liabilities.sum() else 1.0,
        "fixed point residual": float(np.max(np.abs(payments - fixed))) if len(payments) else 0.0,
    }


def network_positions(name, size):
    if name in {"Ring", "Complete"}:
        return nx.circular_layout(range(size))
    if name == "Core-periphery":
        positions = {}
        core = nx.circular_layout(range(4), scale=0.45)
        outer = nx.circular_layout(range(4, size), scale=1.0)
        positions.update(core)
        positions.update(outer)
        return positions
    positions = {}
    left = nx.circular_layout(range(size // 2), scale=0.55, center=(-0.7, 0))
    right = nx.circular_layout(range(size // 2, size), scale=0.55, center=(0.7, 0))
    positions.update(left)
    positions.update(right)
    return positions


def plot_network(relative_liabilities, name, stem, node_values=None):
    graph = nx.DiGraph()
    size = len(relative_liabilities)
    graph.add_nodes_from(range(size))
    for debtor in range(size):
        for creditor in range(size):
            weight = relative_liabilities[debtor, creditor]
            if weight > 1e-12:
                graph.add_edge(debtor, creditor, weight=weight)
    positions = network_positions(name, size)
    values = np.ones(size) if node_values is None else np.asarray(node_values)
    colors = [PALETTE["rose"] if value > 1e-8 else PALETTE["cyan"] for value in values]
    widths = [0.8 + 5 * graph.edges[edge]["weight"] for edge in graph.edges]
    figure, axis = plt.subplots(figsize=(7.2, 6.0), constrained_layout=True)
    nx.draw_networkx_nodes(graph, positions, node_color=colors, node_size=620, edgecolors=PALETTE["text"], linewidths=0.8, ax=axis)
    nx.draw_networkx_labels(graph, positions, font_color=PALETTE["background"], font_weight="bold", ax=axis)
    nx.draw_networkx_edges(graph, positions, width=widths, alpha=0.55, edge_color=PALETTE["text"], arrows=True, arrowsize=13, connectionstyle="arc3,rad=0.06", ax=axis)
    axis.set_title(name)
    axis.set_axis_off()
    save_figure(figure, stem)
    edge_data = pd.DataFrame([{"Debtor": source, "Creditor": target, "Relative liability": graph.edges[source, target]["weight"]} for source, target in graph.edges])
    edge_data.to_csv(FIGURE_DATA / f"{stem}.csv", index=False)


def majorizes(source, target, tolerance=1e-12):
    source_sorted = np.sort(np.asarray(source))[::-1]
    target_sorted = np.sort(np.asarray(target))[::-1]
    return bool(np.all(np.cumsum(source_sorted)[:-1] >= np.cumsum(target_sorted)[:-1] - tolerance) and np.isclose(source_sorted.sum(), target_sorted.sum(), atol=tolerance))

notation = pd.DataFrame([
    {"Symbol": "pbar", "Description": "Total nominal liabilities"},
    {"Symbol": "e", "Description": "External assets"},
    {"Symbol": "Pi", "Description": "Relative liabilities inside the network"},
    {"Symbol": "p", "Description": "Clearing payments"},
    {"Symbol": "pbar - p", "Description": "Unpaid nominal liabilities"},
    {"Symbol": "1 - row sum of Pi", "Description": "Share paid outside the network"},
])
save_table(notation, "clearing_notation", "Notation for the basic proportional clearing model.", "tab:chapter29-clearing-notation", fit=True)

examples = {
    "Two institutions": {
        "external": np.array([1.0, 0.0]),
        "liabilities": np.array([10.0, 10.0]),
        "relative": np.array([[0.0, 0.5], [0.5, 0.0]]),
        "expected": np.array([4.0 / 3.0, 2.0 / 3.0]),
    },
    "Three institution chain": {
        "external": np.array([3.0, 0.0, 0.0]),
        "liabilities": np.array([5.0, 4.0, 2.0]),
        "relative": np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]]),
        "expected": np.array([3.0, 3.0, 2.0]),
    },
}

result_rows = []
fixed_rows = []
flow_rows = []
for example_name, values in examples.items():
    payments = eisenberg_noe_clearing(values["external"], values["liabilities"], values["relative"])
    available = values["external"] + values["relative"].T @ payments
    fixed = np.minimum(values["liabilities"], available)
    assert np.allclose(payments, values["expected"], atol=1e-10)
    for institution in range(len(payments)):
        result_rows.append({
            "Example": example_name,
            "Institution": institution,
            "External assets": values["external"][institution],
            "Nominal liabilities": values["liabilities"][institution],
            "Clearing payment": payments[institution],
            "Unpaid liabilities": values["liabilities"][institution] - payments[institution],
        })
        flow_rows.append({
            "Example": example_name,
            "Institution": institution,
            "External assets": values["external"][institution],
            "Incoming payment": (values["relative"].T @ payments)[institution],
            "Available resources": available[institution],
            "Clearing payment": payments[institution],
        })
    fixed_rows.append({
        "Example": example_name,
        "Maximum fixed point residual": np.max(np.abs(payments - fixed)),
        "Payment bounds pass": bool(np.all(payments >= 0) and np.all(payments <= values["liabilities"] + 1e-12)),
        "Expected vector pass": bool(np.allclose(payments, values["expected"], atol=1e-10)),
    })

exact_results = pd.DataFrame(result_rows)
save_table(exact_results, "exact_network_results", "Hand-checkable clearing payments and unpaid liabilities.", "tab:chapter29-exact-network-results", fit=True)
save_table(pd.DataFrame(fixed_rows), "clearing_fixed_point_checks", "Fixed point and payment-bound checks for the exact examples.", "tab:chapter29-fixed-point-checks", fit=True)
save_table(pd.DataFrame(flow_rows), "payment_flow_decomposition", "External assets, incoming payments, and available resources.", "tab:chapter29-payment-flow", fit=True)

for name, values, stem in [
    ("Two Institutions", examples["Two institutions"], "two_node_clearing_network"),
    ("Three Institution Chain", examples["Three institution chain"], "three_node_clearing_network"),
]:
    payments = eisenberg_noe_clearing(values["external"], values["liabilities"], values["relative"])
    unpaid = values["liabilities"] - payments
    plot_network(values["relative"], name, stem, unpaid)

,Symbol,Description
0,pbar,Total nominal liabilities
1,e,External assets
2,Pi,Relative liabilities inside the network
3,p,Clearing payments
4,pbar - p,Unpaid nominal liabilities
5,1 - row sum of Pi,Share paid outside the network


,Example,Institution,External assets,Nominal liabilities,Clearing payment,Unpaid liabilities
0,Two institutions,0,1,10,1.333333,8.666667
1,Two institutions,1,0,10,0.666667,9.333333
2,Three institution chain,0,3,5,3,2
3,Three institution chain,1,0,4,3,1
4,Three institution chain,2,0,2,2,0


,Example,Maximum fixed point residual,Payment bounds pass,Expected vector pass
0,Two institutions,1.11022e-16,Pass,Pass
1,Three institution chain,0,Pass,Pass


,Example,Institution,External assets,Incoming payment,Available resources,Clearing payment
0,Two institutions,0,1,0.333333,1.333333,1.333333
1,Two institutions,1,0,0.666667,0.666667,0.666667
2,Three institution chain,0,3,0,3,3
3,Three institution chain,1,0,3,3,3
4,Three institution chain,2,0,3,3,2


<Figure size 720x600 with 1 Axes>

<Figure size 720x600 with 1 Axes>

## Independently checking the greatest clearing payment

Starting from promised payments, the monotone update is

$$
p^{(k+1)}=\min\!\left(\overline p,\ e+\Pi^{\mathsf T}p^{(k)}\right),
\qquad p^{(0)}=\overline p.
$$

Here, $p^{(k)}$ is the payment vector after update $k$. Starting from the upper bound makes the sequence nonincreasing. In the basic finite examples below, its limit agrees with both the released RiskLabAI result and a separate linear program that maximizes the sum of feasible payments.

In [2]:
comparison_rows = []
iteration_frames = []
for example_name, values in examples.items():
    released = eisenberg_noe_clearing(values["external"], values["liabilities"], values["relative"])
    iterated, history = greatest_iteration(values["external"], values["liabilities"], values["relative"])
    direct = direct_linear_program(values["external"], values["liabilities"], values["relative"])
    comparison_rows.append({
        "Example": example_name,
        "Iteration steps": len(history) - 1,
        "Released versus iteration residual": np.max(np.abs(released - iterated)),
        "Released versus direct LP residual": np.max(np.abs(released - direct)),
        "Sum of released payments": released.sum(),
        "Pass": bool(np.allclose(released, iterated, atol=1e-10) and np.allclose(released, direct, atol=1e-10)),
    })
    for step, vector in enumerate(history):
        for institution, value in enumerate(vector):
            iteration_frames.append({"Example": example_name, "Step": step, "Institution": institution, "Payment": value})

method_checks = pd.DataFrame(comparison_rows)
save_table(method_checks, "independent_method_checks", "Agreement with monotone iteration and a separate linear program.", "tab:chapter29-independent-method-checks", fit=True)

iteration_data = pd.DataFrame(iteration_frames)
iteration_data.to_csv(FIGURE_DATA / "clearing_iteration_paths.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.6, 4.7), constrained_layout=True)
for axis, example_name in zip(axes, examples):
    part = iteration_data.loc[iteration_data["Example"] == example_name]
    for institution in sorted(part["Institution"].unique()):
        line = part.loc[part["Institution"] == institution]
        axis.plot(line["Step"], line["Payment"], marker="o", linewidth=1.9, label=f"Institution {institution}")
    axis.set_title(example_name)
    axis.set_xlabel("Fixed point update")
    axis.set_ylabel("Payment")
    axis.legend()
    finish_axis(axis)
save_figure(figure, "clearing_iteration_paths")

payment_plot = exact_results[["Example", "Institution", "Nominal liabilities", "Clearing payment"]].copy()
payment_plot.to_csv(FIGURE_DATA / "promised_and_cleared_payments.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.6, 4.7), constrained_layout=True)
for axis, example_name in zip(axes, examples):
    part = payment_plot.loc[payment_plot["Example"] == example_name]
    x = np.arange(len(part))
    axis.bar(x - 0.19, part["Nominal liabilities"], width=0.38, color=PALETTE["amber"], label="Promised")
    axis.bar(x + 0.19, part["Clearing payment"], width=0.38, color=PALETTE["cyan"], label="Cleared")
    axis.set_xticks(x, part["Institution"])
    axis.set_xlabel("Institution")
    axis.set_ylabel("Payment")
    axis.set_title(example_name)
    axis.legend()
    finish_axis(axis)
save_figure(figure, "promised_and_cleared_payments")

,Example,Iteration steps,Released versus iteration residual,Released versus direct LP residual,Sum of released payments,Pass
0,Two institutions,44,5.30687e-13,0,2,Pass
1,Three institution chain,3,0,0,8,Pass


<Figure size 1060x470 with 2 Axes>

<Figure size 1060x470 with 2 Axes>

## Checking clearing-model boundaries

If every monetary input is multiplied by a positive scale $a$, the clearing payments should satisfy

$$
p(ae,a\overline p,\Pi)=a\,p(e,\overline p,\Pi).
$$

The scalar $a$ changes the unit of account, while the relative-liability matrix stays unchanged. This identity is checked at very small, ordinary, and very large scales. Invalid shapes, negative values, nonfinite values, and rows whose internal shares exceed one are rejected explicitly.

In [3]:
base = examples["Two institutions"]
base_payment = eisenberg_noe_clearing(base["external"], base["liabilities"], base["relative"])
scale_rows = []
for scale in [1e-12, 1.0, 1e12]:
    external = base["external"] * scale
    liabilities = base["liabilities"] * scale
    relative = base["relative"].copy()
    before = [external.copy(), liabilities.copy(), relative.copy()]
    payment = eisenberg_noe_clearing(external, liabilities, relative)
    scale_rows.append({
        "Scale": scale,
        "Maximum normalized error": np.max(np.abs(payment / scale - base_payment)),
        "Inputs preserved": bool(all(np.array_equal(current, original) for current, original in zip([external, liabilities, relative], before))),
        "Pass": bool(np.allclose(payment / scale, base_payment, atol=1e-10)),
    })
save_table(pd.DataFrame(scale_rows), "scaling_checks", "Monetary scaling and input-preservation checks.", "tab:chapter29-scaling-checks", fit=True)

invalid_cases = [
    ("Empty system", [], [], []),
    ("Asset vector has two dimensions", [[1.0]], [1.0], [[0.0]]),
    ("Negative external asset", [-1.0], [1.0], [[0.0]]),
    ("Nonfinite liability", [1.0], [np.inf], [[0.0]]),
    ("Internal row sum exceeds one", [1.0], [1.0], [[1.01]]),
]
invalid_rows = []
for case, external, liabilities, relative in invalid_cases:
    try:
        eisenberg_noe_clearing(external, liabilities, relative)
        rejected = False
        error = ""
    except Exception as exc:
        rejected = isinstance(exc, ValueError)
        error = type(exc).__name__
    invalid_rows.append({"Case": case, "Rejected": rejected, "Error": error})
save_table(pd.DataFrame(invalid_rows), "invalid_input_checks", "Explicit rejection checks for invalid clearing inputs.", "tab:chapter29-invalid-inputs", fit=True)

scope = pd.DataFrame([
    {"Question": "Are payments proportional among a debtor's creditors?", "Answer": "Yes", "Boundary": "Fixed relative-liability rows"},
    {"Question": "Are payments capped by nominal liabilities?", "Answer": "Yes", "Boundary": "Limited liability and payment priority"},
    {"Question": "Are bankruptcy costs included?", "Answer": "No", "Boundary": "Basic clearing model only"},
    {"Question": "Are fire sales or price effects included?", "Answer": "No", "Boundary": "No asset-price feedback"},
    {"Question": "Does a link identify a causal spillover?", "Answer": "No", "Boundary": "Mathematical propagation only"},
])
save_table(scope, "clearing_scope_boundaries", "Questions answered and not answered by the basic clearing calculation.", "tab:chapter29-clearing-boundaries", fit=True)

,Scale,Maximum normalized error,Inputs preserved,Pass
0,1e-12,0,Pass,Pass
1,1,0,Pass,Pass
2,"1,000,000,000,000",0,Pass,Pass


,Case,Rejected,Error
0,Empty system,Pass,ValueError
1,Asset vector has two dimensions,Pass,ValueError
2,Negative external asset,Pass,ValueError
3,Nonfinite liability,Pass,ValueError
4,Internal row sum exceeds one,Pass,ValueError


,Question,Answer,Boundary
0,Are payments proportional among a debtor's cre...,Yes,Fixed relative-liability rows
1,Are payments capped by nominal liabilities?,Yes,Limited liability and payment priority
2,Are bankruptcy costs included?,No,Basic clearing model only
3,Are fire sales or price effects included?,No,No asset-price feedback
4,Does a link identify a causal spillover?,No,Mathematical propagation only


,Question,Answer,Boundary
0,Are payments proportional among a debtor's cre...,Yes,Fixed relative-liability rows
1,Are payments capped by nominal liabilities?,Yes,Limited liability and payment priority
2,Are bankruptcy costs included?,No,Basic clearing model only
3,Are fire sales or price effects included?,No,No asset-price feedback
4,Does a link identify a causal spillover?,No,Mathematical propagation only
